In [ ]:
import os
import shutil


def download_stockfish():
    stockfish_dir = "/kaggle/working/stockfish/"
    stockfish_binary = os.path.join(stockfish_dir, "stockfish")
    source_path = "/kaggle/input/stockfish/other/binary/1/stockfish"

    os.makedirs(stockfish_dir, exist_ok=True)

    try:
        if not os.path.exists(stockfish_binary):
            shutil.copy2(source_path, stockfish_binary)
            os.chmod(stockfish_binary, 0o755)
        return stockfish_binary
    except Exception as e:
        print(f"Error setting up Stockfish: {e}")
        return None


STOCKFISH_PATH = download_stockfish()

In [ ]:
from dataclasses import dataclass


@dataclass
class TrainingConfig:
    hidden_size: int = 256
    intermediate_size: int = 1024
    num_hidden_layers: int = 6
    num_attention_heads: int = 8
    max_position_embeddings: int = 512
    max_length: int = 512

    batch_size: int = 96
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    warmup_steps: int = 500
    lr_schedule: str = "cosine"

    train_epochs: int = 6

    max_positions: int = 8192
    train_split: float = 0.9

    stockfish_path: str = ""
    stockfish_depth: int = 10
    stockfish_num_workers: int = 16

    log_every_n_batches: int = 50

    save_dir: str = "chess_model"


config = TrainingConfig()
config.stockfish_path = STOCKFISH_PATH  # type: ignore

In [ ]:
from typing import List, Optional
import json


class ChessTokenizer:
    def __init__(self):
        self.special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
        self.vocab = self._build_vocab()
        self.token_to_id = {token: idx for idx, token in enumerate(self.vocab)}
        self.id_to_token = {idx: token for token, idx in self.token_to_id.items()}

        self.pad_token = "<PAD>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"
        self.unk_token = "<UNK>"

        self.pad_token_id = self.token_to_id[self.pad_token]
        self.bos_token_id = self.token_to_id[self.bos_token]
        self.eos_token_id = self.token_to_id[self.eos_token]
        self.unk_token_id = self.token_to_id[self.unk_token]

    def _build_vocab(self) -> List[str]:
        vocab = []
        vocab.extend(self.special_tokens)

        files = "abcdefgh"
        ranks = "12345678"
        squares = [f + r for f in files for r in ranks]

        for from_sq in squares:
            for to_sq in squares:
                vocab.append(f"{from_sq}{to_sq}")

                if from_sq[1] == "7" and to_sq[1] == "8":
                    for piece in ["q", "r", "b", "n"]:
                        vocab.append(f"{from_sq}{to_sq}{piece}")

                if from_sq[1] == "2" and to_sq[1] == "1":
                    for piece in ["q", "r", "b", "n"]:
                        vocab.append(f"{from_sq}{to_sq}{piece}")

        return vocab

    def encode_move(self, move_uci: str) -> str:
        return move_uci

    def decode_move(self, move_token: str) -> Optional[str]:
        if move_token in self.token_to_id and len(move_token) >= 4:
            return move_token
        return None

    def encode(self, moves: List[str]) -> List[int]:
        tokens = [self.bos_token]
        tokens.extend(moves)
        tokens.append(self.eos_token)

        return [self.token_to_id.get(token, self.unk_token_id) for token in tokens]

    def decode(self, token_ids: List[int]) -> List[str]:
        return [self.id_to_token.get(id, self.unk_token) for id in token_ids]

    def __len__(self):
        return len(self.vocab)

    def save_pretrained(self, path: str):
        os.makedirs(path, exist_ok=True)

        vocab_data = {
            "vocab": self.vocab,
            "special_tokens": self.special_tokens,
        }

        with open(os.path.join(path, "vocab.json"), "w") as f:
            json.dump(vocab_data, f, indent=2)

        config = {
            "pad_token": self.pad_token,
            "bos_token": self.bos_token,
            "eos_token": self.eos_token,
            "unk_token": self.unk_token,
        }

        with open(os.path.join(path, "tokenizer_config.json"), "w") as f:
            json.dump(config, f, indent=2)


tokenizer = ChessTokenizer()

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Example encoding: {tokenizer.encode(['e2e4', 'e7e5'])}")

In [ ]:
from torch.utils.data import Dataset
from typing import Dict
import torch


class ChessPositionDataset(Dataset):
    def __init__(
        self,
        position_samples: List[Dict],
        tokenizer: ChessTokenizer,
        max_length: int = 512,
    ):
        self.position_samples = position_samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.position_samples)

    def __getitem__(self, idx):
        sample = self.position_samples[idx]
        moves = sample["moves"]
        target_move = sample["target_move"]
        position_idx = sample["position_idx"]

        input_ids = self.tokenizer.encode(moves)

        if len(input_ids) > self.max_length:
            input_ids = input_ids[: self.max_length]

        input_ids = input_ids + [self.tokenizer.pad_token_id] * (
            self.max_length - len(input_ids)
        )

        attention_mask = [
            1 if id != self.tokenizer.pad_token_id else 0 for id in input_ids
        ]

        target_token_id = self.tokenizer.token_to_id.get(
            target_move, self.tokenizer.unk_token_id
        )

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "target": torch.tensor(target_token_id, dtype=torch.long),
            "position_idx": position_idx,
        }


def collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "targets": torch.stack([item["target"] for item in batch]),
        "position_indices": [item["position_idx"] for item in batch],
    }

In [ ]:
from typing import Tuple
from torch.utils.data import DataLoader
import random


def create_dataloaders(
    position_samples: List[Dict],
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
) -> Tuple[DataLoader, DataLoader]:
    random.shuffle(position_samples)
    split_idx = int(config.train_split * len(position_samples))
    train_samples = position_samples[:split_idx]
    val_samples = position_samples[split_idx:]

    print(f"Train positions: {len(train_samples)}, Val positions: {len(val_samples)}")

    train_dataset = ChessPositionDataset(train_samples, tokenizer, config.max_length)
    val_dataset = ChessPositionDataset(val_samples, tokenizer, config.max_length)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    return train_loader, val_loader


print("DataLoader creation functions defined")

In [ ]:
import torch.nn as nn
from transformers import LlamaConfig, LlamaForCausalLM


class ChessTransformer(nn.Module):
    def __init__(self, config: LlamaConfig, vocab_size: int):
        super().__init__()

        config.vocab_size = vocab_size
        self.config = config

        self.model = LlamaForCausalLM(config)

    def forward(self, input_ids, attention_mask=None, labels=None):
        if labels is not None:
            outputs = self.model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            return {"loss": outputs.loss, "logits": outputs.logits}
        else:
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            return {"loss": None, "logits": outputs.logits}

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from typing import List, Dict

import chess
import chess.pgn
import chess.engine

import torch
import pickle
from tqdm import tqdm


def _evaluate_position(args):
    stockfish_path, board_fen, depth, legal_moves_uci = args

    try:
        engine = chess.engine.SimpleEngine.popen_uci(stockfish_path)
        engine.configure({"Threads": 1})

        board = chess.Board(board_fen)
        scores = {}

        for move_uci in legal_moves_uci:
            temp_board = board.copy()
            move = chess.Move.from_uci(move_uci)
            temp_board.push(move)

            try:
                info = engine.analyse(
                    temp_board, chess.engine.Limit(depth=depth), multipv=1
                )
                score = info["score"].relative.score(mate_score=10000)
                scores[move_uci] = -score
            except:
                scores[move_uci] = 0

        engine.quit()
        return board_fen, scores
    except:
        return board_fen, {}


def extract_and_evaluate_positions(
    games: List[Dict],
    stockfish_path: str,
    depth: int,
    num_workers: int,
    max_positions: int,
    batch_size: int = 100,
    cache_path: str = "position_evaluations.pkl",
) -> List[Dict]:

    if os.path.exists(cache_path):
        print(f"Loading cached position samples from {cache_path}")
        with open(cache_path, "rb") as f:
            position_samples = pickle.load(f)
        print(f"Loaded {len(position_samples)} cached position samples")
        return position_samples[:max_positions]

    print(f"Extracting up to {max_positions} positions from games...")

    position_samples = []
    current_batch_positions = []
    current_batch_metadata = []

    print("Extracting and evaluating positions in batches...")

    for game_idx, game in enumerate(tqdm(games, desc="Processing games")):
        if len(position_samples) >= max_positions:
            break

        board = chess.Board()
        moves = game["moves"]

        for pos_idx, move_uci in enumerate(moves):
            if len(position_samples) >= max_positions:
                break

            try:
                if not board.is_game_over() and pos_idx > 0:
                    position_fen = board.fen()
                    legal_moves_uci = [m.uci() for m in board.legal_moves]

                    current_batch_positions.append(
                        (stockfish_path, position_fen, depth, legal_moves_uci)
                    )
                    current_batch_metadata.append(
                        {
                            "moves": moves[:pos_idx],
                            "position_idx": pos_idx,
                            "fen": position_fen,
                        }
                    )

                    if len(current_batch_positions) >= batch_size:
                        batch_samples = _evaluate_batch(
                            current_batch_positions, current_batch_metadata, num_workers
                        )
                        position_samples.extend(batch_samples)

                        current_batch_positions = []
                        current_batch_metadata = []

                        if len(position_samples) >= max_positions:
                            position_samples = position_samples[:max_positions]
                            break

                move = chess.Move.from_uci(move_uci)
                if move in board.legal_moves:
                    board.push(move)
                else:
                    break
            except:
                break

    if current_batch_positions and len(position_samples) < max_positions:
        batch_samples = _evaluate_batch(
            current_batch_positions, current_batch_metadata, num_workers
        )
        position_samples.extend(batch_samples)
        position_samples = position_samples[:max_positions]

    print(f"Successfully created {len(position_samples)} position samples")

    with open(cache_path, "wb") as f:
        pickle.dump(position_samples, f)
    print(f"Saved position samples to {cache_path}")

    return position_samples


def _evaluate_batch(positions_to_evaluate, position_metadata, num_workers):
    batch_samples = []

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        results = list(executor.map(_evaluate_position, positions_to_evaluate))

        for (fen, scores), metadata in zip(results, position_metadata):
            if scores:
                best_move = max(scores.items(), key=lambda x: x[1])[0]
                batch_samples.append(
                    {
                        "moves": metadata["moves"],
                        "target_move": best_move,
                        "position_idx": metadata["position_idx"],
                        "fen": fen,
                    }
                )

    return batch_samples

In [ ]:
import pandas as pd
import io


def load_dataset(csv_path):
    print(f"Loading dataset from {csv_path}...")
    df = pd.read_csv(csv_path)

    print(f"Total games in CSV: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")

    return df


def process_games(
    df: pd.DataFrame, max_positions: int, avg_positions_per_game: int = 20
) -> List[Dict]:
    samples = []

    estimated_games_needed = int(max_positions / avg_positions_per_game * 1.2)
    games_to_process = min(estimated_games_needed, len(df))

    print(
        f"Processing up to {games_to_process} games to get {max_positions} positions..."
    )

    for idx in range(games_to_process):
        if idx % 1000 == 0:
            print(
                f"Processed {idx}/{games_to_process} games, extracted {len(samples)} games so far"
            )

        try:
            pgn_text = df.iloc[idx]["pgn"]

            game = chess.pgn.read_game(io.StringIO(pgn_text))
            if game is None:
                continue

            moves = [move.uci() for move in game.mainline_moves()]

            if len(moves) >= 10:
                samples.append({"moves": moves})

        except Exception as e:
            continue

    print(f"Extracted {len(samples)} games.")
    return samples

In [ ]:
def create_model(
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
    device: torch.device,
) -> ChessTransformer:

    model_config = LlamaConfig(
        vocab_size=len(tokenizer),
        hidden_size=config.hidden_size,
        intermediate_size=config.intermediate_size,
        num_hidden_layers=config.num_hidden_layers,
        num_attention_heads=config.num_attention_heads,
        max_position_embeddings=config.max_position_embeddings,
        rms_norm_eps=1e-5,
        initializer_range=0.02,
        use_cache=False,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    model = ChessTransformer(model_config, len(tokenizer))

    model = model.to(device)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = nn.DataParallel(model)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    return model


def save_checkpoint(
    model: ChessTransformer,
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
    epoch: int,
    metrics: Dict[str, float],
):
    save_path = os.path.join(config.save_dir, f"epoch_{epoch}")
    os.makedirs(save_path, exist_ok=True)

    model_to_save = model.module if isinstance(model, nn.DataParallel) else model

    model_to_save.model.save_pretrained(save_path, safe_serialization=True)

    tokenizer.save_pretrained(save_path)

    with open(os.path.join(save_path, "training_metadata.json"), "w") as f:
        json.dump(
            {
                "epoch": epoch,
                "metrics": metrics,
            },
            f,
            indent=2,
        )

    print(f"Checkpoint saved to {save_path}")

In [ ]:
from tqdm import tqdm
import torch
import torch.nn.functional as F


def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        return max(0.0, 0.5 * (1.0 + torch.cos(torch.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    device,
):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_predictions = 0

    progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Training")

    for _, batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)
        position_indices = batch["position_indices"]

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        if isinstance(outputs, dict):
            logits = outputs["logits"]
        else:
            logits = outputs.logits

        if logits.dim() == 4:
            logits = logits.squeeze(0)

        selected_logits = []
        for i, pos in enumerate(position_indices):
            selected_logits.append(logits[i, pos, :])

        selected_logits = torch.stack(selected_logits)

        loss = F.cross_entropy(
            selected_logits.contiguous(),
            targets.contiguous(),
        )

        predicted = torch.argmax(selected_logits, dim=-1)
        batch_correct = (predicted == targets).sum().item()
        batch_predictions = len(targets)

        total_correct += batch_correct
        total_predictions += batch_predictions

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        progress_bar.set_postfix(
            {
                "loss": f"{loss.item():.4f}",
                "acc": f"{batch_correct/max(batch_predictions, 1):.3f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            }
        )

    avg_loss = total_loss / len(dataloader) if total_loss > 0 else 0.0
    avg_accuracy = total_correct / total_predictions if total_predictions > 0 else 0.0

    return {
        "loss": avg_loss,
        "accuracy": avg_accuracy,
        "total_predictions": total_predictions,
    }


def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_predictions = 0

    progress_bar = tqdm(dataloader, desc="Evaluating")

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)
            position_indices = batch["position_indices"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            if isinstance(outputs, dict):
                logits = outputs["logits"]
            else:
                logits = outputs.logits

            if logits.dim() == 4:
                logits = logits.squeeze(0)

            selected_logits = []
            for i, pos in enumerate(position_indices):
                selected_logits.append(logits[i, pos, :])

            selected_logits = torch.stack(selected_logits)

            loss = F.cross_entropy(selected_logits.contiguous(), targets.contiguous())

            predicted = torch.argmax(selected_logits, dim=-1)
            batch_correct = (predicted == targets).sum().item()
            batch_predictions = len(targets)

            total_loss += loss.item()
            total_correct += batch_correct
            total_predictions += batch_predictions

            progress_bar.set_postfix(
                {
                    "loss": f"{loss.item():.4f}",
                    "acc": f"{batch_correct/batch_predictions:.3f}",
                }
            )

    avg_loss = total_loss / len(dataloader) if total_loss > 0 else 0.0
    avg_accuracy = total_correct / total_predictions if total_predictions > 0 else 0.0

    return {
        "loss": avg_loss,
        "accuracy": avg_accuracy,
        "total_predictions": total_predictions,
    }

In [ ]:
df = load_dataset(
    csv_path="/kaggle/input/chesscom-user-games-60000-games/club_games_data.csv",
)
print(f"\nDataset shape: {df.shape}")
if "pgn" in df.columns:
    print(f"First game preview:\n{df['pgn'].iloc[0][:200]}...")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Available GPUs: {torch.cuda.device_count()}")

In [ ]:
print("\n" + "=" * 50)
print("TRAINING")
print("=" * 50)

games = process_games(df, config.max_positions)

position_samples = extract_and_evaluate_positions(
    games,
    config.stockfish_path,
    config.stockfish_depth,
    config.stockfish_num_workers,
    config.max_positions,
    batch_size=100,
)

train_loader, val_loader = create_dataloaders(position_samples, tokenizer, config)

model = create_model(tokenizer, config, device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

total_steps = len(train_loader) * config.train_epochs
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=config.warmup_steps, num_training_steps=total_steps
)

for epoch in range(config.train_epochs):
    print(f"\nEpoch {epoch + 1}/{config.train_epochs}")
    print("-" * 50)

    train_metrics = train_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device,
    )
    print(f"Train Loss: {train_metrics['loss']:.4f}")
    print(f"Train Accuracy: {train_metrics['accuracy']:.4f}")
    print(f"Train Positions: {train_metrics['total_predictions']}")

    val_metrics = evaluate(model, val_loader, device)
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Val Positions: {val_metrics['total_predictions']}")

    save_checkpoint(model, tokenizer, config, epoch + 1, val_metrics)